In [1]:
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F

class SpatialClusterHead(nn.Module):
    def __init__(self, feature_dim, num_clusters):
        super().__init__()
        self.spatial_mlp = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, num_clusters)
        )
        self.feature_mlp = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_clusters)
        )
        
        # THE FIX: Equalizers
        # This prevents the feature_mlp from bullying the spatial_mlp
        self.spatial_norm = nn.LayerNorm(num_clusters)
        self.feature_norm = nn.LayerNorm(num_clusters)
        
    def forward(self, X_combined, spatial_weight=3.0):
        spatial_logits = self.spatial_mlp(X_combined[:, :2])
        feature_logits = self.feature_mlp(X_combined)
        
        # Normalize both to unit variance before combining
        s = spatial_logits / (spatial_logits.std() + 1e-8)
        f = feature_logits / (feature_logits.std() + 1e-8)
        
        
        return spatial_weight * s + f

class FirstTerm(nn.Module):
    def __init__(self, num_cell_types, num_of_clusters , embedding_dim=8):
        super().__init__()
        self.cell_embedding = nn.Embedding(num_cell_types, embedding_dim)
        # Initialize 10.9M weights (The Dials)
        feature_dim = 18 + embedding_dim
        self.edge_mlp = nn.Sequential(
            nn.Linear(feature_dim*2, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

        nn.init.normal_(self.edge_mlp[2].weight, mean=0.0, std=0.001)
        nn.init.constant_(self.edge_mlp[2].bias, 0.01)


        # 3. THE DUAL-PATH CLUSTER HEAD
        # Replaces the standard Sequential MLP
        self.cluster_head = SpatialClusterHead(
            feature_dim=feature_dim, 
            num_clusters=num_of_clusters
        )



    def forward(self, X, X_cell_ids, num_nodes, p_indices, A_skip_csr, current_k, tau=1.0):
        
        X_cell_ids = X_cell_ids.squeeze()
        cell_features = self.cell_embedding(X_cell_ids)  # Shape: [num_nodes, embedding_dim]
        X_combined = torch.cat([X, cell_features], dim=1)  # Shape: [num_nodes, 18 + embedding_dim]

        src_features = X_combined[p_indices[0]]  # Shape: [num_edges, feature_dim]
        dst_features = X_combined[p_indices[1]]  # Shape: [num_edges, feature_dim]
        edge_inputs = torch.cat([src_features, dst_features], dim=1)  # Shape: [num_edges, feature_dim*2]

        dynamic_p_weights = self.edge_mlp(edge_inputs).squeeze(-1)
        safe_weights = F.softplus(dynamic_p_weights) 


        


        # Enforce P >= 0 and build sparse matrix
        P = torch.sparse_coo_tensor(p_indices, safe_weights, 
                                    (num_nodes, num_nodes)).coalesce()
        
        # Reconstruction: XP
        X_hat = torch.sparse.mm(P, X)
        
        # Loss: ||X - XP||
        error = X - X_hat
        loss1 = torch.mean(error**2)   

        # Pass all node features through the head
        logits = self.cluster_head(X_combined) # Shape: [n, k]

        logits = logits[:, :current_k]
        
        #TERM2
        #C matrix with probability distribution across clusters for each node
        C = F.gumbel_softmax(logits, tau=tau, hard=False) # Shape: [n, k]
        # Inside FirstTerm.forward or the training loop
        # tau_smooth = 0.6  # Higher = more spread out/soft
        # C = F.softmax(logits / tau_smooth, dim=-1)
        # C = F.softmax(logits, dim=-1)  # Ensure positivity for SDDMM

        p_vals = P.values()
        
        # 1. Sum across rows (dim=1) to get the total weight leaving each node
        row_sums = torch.sparse.sum(P, dim=1).to_dense()
        
        # 2. Expand row_sums to match the non-zero values 
        # P.indices()[0] contains the row index for every specific edge
        p_vals_norm = p_vals / (row_sums[P.indices()[0]] + 1e-8)
        
        # Rebuild using the exact same sorted indices
        P_norm = torch.sparse_coo_tensor(P.indices(), p_vals_norm, 
                                         (num_nodes, num_nodes)).coalesce()
        
        # 2. Convert to CSR format (Required for the CUDA SDDMM engine)
        P_csr = P_norm.to_sparse_csr()
        
        # 3. SDDMM Magic! 
        # beta=1.0, alpha=-1.0 calculates exactly: (1.0 * P_csr) - (1.0 * C @ C^T)
        # It ONLY calculates this at the 10.9M non-zero locations!
        diff_csr = torch.sparse.sampled_addmm(P_csr, C, C.t(), beta=1.0, alpha=-1.0)
        
        # 4. Square the differences and sum them
        loss2 = torch.sum(diff_csr.values() ** 2)


        #TERM3
        M = torch.matmul(C.t(), torch.sparse.mm(A_skip_csr, C))  # [k, n] @ [n, n] @ [n, k] -> [k, k]
        # 2. Normalize M into a probability distribution (M_tilde)
        M = torch.clamp(M, min=0)
        M_tilde = M / (M.sum() + 1e-8)
        loss3 = -torch.sum(M_tilde * torch.log(M_tilde + 1e-8))

        # 3. Calculate Shannon Entropy: -sum(p * log(p))
        # We only calculate for non-zero entries to avoid log(0)
        # loss3 = torch.sum(M_tilde * torch.log(M_tilde + 1e-8))

        

        alpha_1 = 1.0 
        alpha_2 = 1.0    
        alpha_3 = 1.0    
        loss = alpha_1* loss1 + (alpha_2 * loss2) +(alpha_3 * loss3) 



        return  loss, loss1 , loss2, loss3,  C , X_combined


import math
def get_tau(epoch):
    tau_start = 2.0
    tau_mid = 1.35   # Targets ~25% Confidence for the middle flatline
    tau_end = 0.85   # Targets ~40% Confidence for the final floor

    if epoch < 75:
        # Phase 1: Smooth descent to the middle flatline
        progress = epoch / 75.0
        return tau_mid + 0.5 * (tau_start - tau_mid) * (1 + math.cos(math.pi * progress))
        
    elif epoch < 150:
        # Phase 2: THE MIDDLE FLATLINE (Epoch 75 to 150)
        return tau_mid
        
    elif epoch < 200:
        # Phase 3: Smooth descent to the final floor
        progress = (epoch - 150) / 50.0
        return tau_end + 0.5 * (tau_mid - tau_end) * (1 + math.cos(math.pi * progress))
        
    else:
        # Phase 4: THE FINAL FLATLINE (Epoch 200 to 250+)
        return tau_end

In [2]:
#Graph Neural Network Inference on the compressed graph (X_tilde, A_tilde_skip) 

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GraphConv, to_hetero , global_mean_pool



# ==========================================
# 1. The Base Homogeneous SAGE Model
# ==========================================
class BaseSAGE(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        # Using (-1, -1) leverages PyG's lazy initialization 
        self.conv1 = GraphConv((-1, -1), hidden_dim)
        self.conv2 = GraphConv((-1, -1), hidden_dim)

    def forward(self, x, edge_index, edge_weight=None):
        x = F.elu(self.conv1(x, edge_index, edge_weight))
        x = F.elu(self.conv2(x, edge_index, edge_weight))
        return x

class HeteroCTS_GNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_params):
        super().__init__()
        
        # 1. Define the Heterogeneous Schema explicitly
        self.metadata = (
            ['supernode'], 
            [
                ('supernode', 'physical', 'supernode'), 
                ('supernode', 'timing', 'supernode')
            ]
        )
        
        # 2. Initial projection to get raw features into hidden_dim space
        self.proj = nn.Linear(input_dim, hidden_dim)
        
        # 3. Create the Heterogeneous GNN
        # 'sum' aggregation combines the messages from physical and timing paths at each node
        self.gnn = to_hetero(BaseSAGE(hidden_dim), self.metadata, aggr='sum')
        
        # 4. Task Heads 
        self.power_head = nn.Sequential(
            nn.Linear(hidden_dim + num_params, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        # Wirelength: Expects mean + variance concatenated (size: 2 * hidden_dim)
        self.wl_head = nn.Sequential(
            nn.Linear(hidden_dim + num_params, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )  
        
        # Skew: Placeholder for Attention Pooling (Currently using Mean)
        self.skew_attn = nn.Linear(hidden_dim, 1) # Attention weights
        self.skew_head = nn.Sequential(
            nn.Linear(hidden_dim + num_params, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )


    def forward(self, x_dict, edge_index_dict,cts_params, edge_weight_dict=None):
        # Initial projection and GNN message passing
        x = {key: self.proj(feat) for key, feat in x_dict.items()}
        h_dict = self.gnn(x, edge_index_dict, edge_weight_dict)
        h = h_dict['supernode'] # [K, hidden_dim]

        # 2. Power: Mean Pool and Concatenate Knobs
        h_power = torch.mean(h, dim=0, keepdim=True) # [1, hidden_dim]
        h_power_combined = torch.cat([h_power, cts_params], dim=-1)
        power_pred = self.power_head(h_power_combined)

        # 3. Wirelength: Mean + Var Pool and Concatenate Knobs [cite: 198, 199]
        mu = h.mean(dim=0, keepdim=True)
        h_wl_combined = torch.cat([mu,  cts_params], dim=-1)
        wl_pred = self.wl_head(h_wl_combined)

        # 4. Skew: Attention Pool and Concatenate Knobs [cite: 190, 191]
        attn_scores = F.softmax(self.skew_attn(h), dim=0)
        h_skew = torch.sum(attn_scores * h, dim=0, keepdim=True)
        h_skew_combined = torch.cat([h_skew, cts_params], dim=-1)
        skew_pred = self.skew_head(h_skew_combined)

        return skew_pred, power_pred, wl_pred
    
 

/home/rain/CTS-Task-Aware-Clustering/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os , random
processed_dir = "processed_graphs"
# List all processed file names
design_files = [f for f in os.listdir(processed_dir) if f.endswith('.pt')]

grouped_designs = {}
for f in design_files:
    # Splits "aes_run_20260306..." to extract just "aes"
    base_name = f.split('_run_')[0] 
    if base_name not in grouped_designs:
        grouped_designs[base_name] = []
    grouped_designs[base_name].append(f)

# 2. Pick exactly 5 from each category
balanced_files = []
for name, files in grouped_designs.items():
    random.shuffle(files)          # Shuffle within the category
    selected = files[:1]           # Take exactly 5 (or less if a category has < 5)
    balanced_files.extend(selected)

# 3. Final shuffle so the epoch isn't ordered by chip type
random.shuffle(balanced_files)
design_files = balanced_files      # Overwrite the main list

print(f"🔥 Fast-Tracking on {len(design_files)} perfectly balanced designs!")
# Quick check to prove it's balanced
print("Balance Check:", {name: len([x for x in design_files if name in x]) for name in grouped_designs.keys()})


🔥 Fast-Tracking on 4 perfectly balanced designs!
Balance Check: {'ethmac': 1, 'aes': 1, 'picorv32': 1, 'sha256': 1}


In [4]:
from helper import load_cts_parameters, get_compressed_graph, relative_masking

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
csv_file = "./dataset_with_def/unified_manifest_normalized.csv"

# ==========================================
    # 1. LOAD AND FREEZE PHASE 1
    # ==========================================
print("❄️ Loading Phase 1 Brain...")
phase1_model = FirstTerm(num_cell_types=425, num_of_clusters=1800).to(device)
phase1_model.load_state_dict(torch.load("phase1_clustering_ep250_twohop.pt", map_location=device))
phase1_model.eval()

import torch.optim as optim

# ==========================================
# INITIALIZE GNN (Check your X_tilde dimension! Usually it's ~20 features)
# ==========================================
# Let's assume X_tilde has 20 columns. Adjust this if your raw X has a different size!
gnn_model = HeteroCTS_GNN(
    input_dim=28,       # <-- Make sure this matches X_tilde.shape[1]
    hidden_dim=64, 
    num_params=4        # <-- 4 CTS knobs
).to(device)

optimizer = optim.Adam(gnn_model.parameters(), lr=0.001)
criterion = nn.MSELoss()

print("\n🚀 Starting On-The-Fly GNN Training...")

epochs = 1
for epoch in range(epochs):
    gnn_model.train()

    for filename in design_files:
        placement_id = filename.replace('.pt', '')
        data = torch.load(os.path.join(processed_dir, filename), map_location=device)
        
        # 1. Get targets from CSV
        cts_runs = load_cts_parameters(csv_file, placement_id, device)
        if not cts_runs:
            continue
            
        # 2. ON-THE-FLY COMPRESSION (Frozen Phase 1)
        with torch.no_grad():
            _, _, _, _, C, X_combined = phase1_model(
                data['X'], data['X_cell_ids'].squeeze(), data['num_nodes'], 
                data['p_indices'], data['A_skip_csr'], data['current_k'], tau=0.85
            )


            
            X_tilde, A_tilde_skip, A_tilde_wire = get_compressed_graph(
                X_combined, C, data['A_skip_csr'], data['A_wire_csr'], data['raw_areas']
            )

            # ==========================================
            # 🔬 INSPECTING X_TILDE SUPER-FEATURES
            # ==========================================
            feature_names = [
                "Normalized X", "Normalized Y", 
                "Dist Boundary L", "Dist Boundary R", "Dist Boundary B", "Dist Boundary T",
                "Log Cell Area", "Avg Pin Cap", "Total Pin Cap", "Log Drive Strength",
                "Is Sequential (Ratio)", "Is Buffer (Ratio)", 
                "Toggle Count", "Sum Toggle Count", "Signal Prob", "Non-Zero Count",
                "Log Fan-In", "Log Fan-Out"
            ]

            print("\n" + "="*60)
            print(f"🔬 INSPECTING X_TILDE (Shape: {X_tilde.shape})")
            print("="*60)
            
            for i in range(X_tilde.shape[1]):
                col_data = X_tilde[:, i]
                
                # Dynamically map the names
                if i < len(feature_names):
                    name = feature_names[i]
                elif i == X_tilde.shape[1] - 2:
                    name = "⭐ Cluster Mass (Raw Area Sum)"
                elif i == X_tilde.shape[1] - 1:
                    name = "⭐ Cluster Spread (Spatial Variance)"
                else:
                    # Just in case X_combined has extra embedding dimensions attached!
                    name = f"Phase 1 Hidden Embedding {i}"
                    
                print(f"[{i:2d}] {name:<35}: Min {col_data.min().item():>8.4f} | Max {col_data.max().item():>8.4f} | Mean {col_data.mean().item():>8.4f}")
            
            print("="*60 + "\n")
            
            # Get BOTH the indices and the weights from your masking function!
            wire_edge_idx, wire_edge_weight = relative_masking(A_tilde_wire, threshold=0.10)
            skip_edge_idx, skip_edge_weight = relative_masking(A_tilde_skip, threshold=0.10)
            
        # # ==========================================
        # # Construct the Hetero Dictionaries
        # # ==========================================
        # x_dict = {
        #     'supernode': X_tilde
        # }
        
        # edge_index_dict = {
        #     ('supernode', 'physical', 'supernode'): wire_edge_idx,
        #     ('supernode', 'timing', 'supernode'): skip_edge_idx
        # }
        
        # edge_weight_dict = {
        #     ('supernode', 'physical', 'supernode'): wire_edge_weight,
        #     ('supernode', 'timing', 'supernode'): skip_edge_weight
        # }







❄️ Loading Phase 1 Brain...

🚀 Starting On-The-Fly GNN Training...


/home/rain/CTS-Task-Aware-Clustering/venv/lib/python3.12/site-packages/torch/_utils.py:361: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  result = torch.sparse_compressed_tensor(



🔬 INSPECTING X_TILDE (Shape: torch.Size([798, 28]))
[ 0] Normalized X                       : Min  -1.7092 | Max   1.7420 | Mean  -0.0026
[ 1] Normalized Y                       : Min  -1.5560 | Max   1.5843 | Mean  -0.0118
[ 2] Dist Boundary L                    : Min  -1.7092 | Max   1.7420 | Mean  -0.0026
[ 3] Dist Boundary R                    : Min  -1.7420 | Max   1.7092 | Mean   0.0026
[ 4] Dist Boundary B                    : Min  -1.5843 | Max   1.5560 | Mean   0.0118
[ 5] Dist Boundary T                    : Min  -1.5560 | Max   1.5843 | Mean  -0.0118
[ 6] Log Cell Area                      : Min  -1.2071 | Max   0.9238 | Mean  -0.0014
[ 7] Avg Pin Cap                        : Min  -0.5231 | Max   2.1483 | Mean   0.0007
[ 8] Total Pin Cap                      : Min  -0.8516 | Max   1.2806 | Mean  -0.0005
[ 9] Log Drive Strength                 : Min  -0.3534 | Max   0.9896 | Mean   0.0047
[10] Is Sequential (Ratio)              : Min   0.0239 | Max   0.6142 | Mean   0.2214
[